In [8]:
import requests
import re
import time
from pathlib import Path
from typing import List, Dict
from urllib.parse import urljoin
from dataclasses import dataclass
from bs4 import BeautifulSoup
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm import tqdm
from firecrawl import FirecrawlApp, ScrapeOptions
import re
from urllib.parse import urlparse
import re
from pathlib import Path
from typing import List
from googleapiclient.discovery import build
from google.oauth2 import service_account
from typing import List, Tuple


In [9]:
FIRECRAWL_API="fc-e64960c290c04b96b6fdc1926d51bb8e"

CREDENTIALS_FILE = "C:/Users/apwbm/OneDrive/Desktop/PROJECTS/P1/LEAD ENRICHMENT/lead-enricher-ai-be/data/url-to-email-445616-cebe4868914f.json"
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc/edit?gid=0#gid=0" 

# - Column C: Website URLs (starting from row 2)
# - Column K: Will be filled with scraped content
# - Column R: Will be filled with status information

# if __name__ == "__main__":
#     results = scrape_from_google_sheet(CREDENTIALS_FILE, GOOGLE_SHEET_URL)

In [10]:
class GoogleSheetsReader:
    """Reads URLs from Google Sheets."""

    def __init__(self, credentials_file: str):
        print("🔄 Initializing Google Sheets Reader...")
        self.credentials_file = credentials_file
        self.service = self._setup_service()
        print("✅ Google Sheets service initialized")

    def _setup_service(self):
        """Initialize Google Sheets API service."""
        print("🔧 Setting up Google Sheets API service...")
        if not Path(self.credentials_file).exists():
            raise FileNotFoundError(f"❌ Credentials file not found: {self.credentials_file}")
        
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(
            self.credentials_file, scopes=scopes
        )
        return build('sheets', 'v4', credentials=creds)

    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        """Extract spreadsheet ID from URL."""
        print(f"📊 Extracting spreadsheet ID from URL: {sheet_url}")
        match = re.search(r'/spreadsheets/d/([a-zA-Z0-9-_]+)', sheet_url)
        if match:
            spreadsheet_id = match.group(1)
            print(f"✅ Spreadsheet ID extracted: {spreadsheet_id}")
            return spreadsheet_id
        raise ValueError(f"❌ Invalid Google Sheet URL: {sheet_url}")

    def get_urls(self, spreadsheet_id: str, range_name: str = "C2:C") -> List[Tuple[int, str]]:
        """Retrieve URLs with their row indices from a specified column range."""
        print(f"📋 Fetching URLs from Google Sheet (Spreadsheet ID: {spreadsheet_id}, Range: {range_name})...")
        try:
            result = self.service.spreadsheets().values().get(
                spreadsheetId=spreadsheet_id,
                range=range_name
            ).execute()
            values = result.get('values', [])
            start_row = int(range_name.split(':')[0][1:]) if range_name[1].isdigit() else 2
            urls_with_rows = [(start_row + i, row[0]) for i, row in enumerate(values) if row and row[0].strip()]
            print(f"✅ Found {len(urls_with_rows)} URLs to scrape")
            return urls_with_rows
        except Exception as e:
            print(f"❌ Error fetching URLs: {e}")
            return []

In [11]:
reader = GoogleSheetsReader(CREDENTIALS_FILE)
spreadsheet_id = reader.extract_spreadsheet_id(GOOGLE_SHEET_URL)
main_urls_with_rows = reader.get_urls(spreadsheet_id, range_name="C2:C")

🔄 Initializing Google Sheets Reader...
🔧 Setting up Google Sheets API service...
✅ Google Sheets service initialized
📊 Extracting spreadsheet ID from URL: https://docs.google.com/spreadsheets/d/1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc/edit?gid=0#gid=0
✅ Spreadsheet ID extracted: 1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc
📋 Fetching URLs from Google Sheet (Spreadsheet ID: 1lmVQ8jnKWYowsfEG89FiiLIEAIkBs7uhshuOvMQ1tZc, Range: C2:C)...
✅ Found 2 URLs to scrape


In [12]:
class Config:
    """Configuration settings for scraping."""
    max_retries: int = 3
    request_timeout: int = 30
    delay_between_requests: float = 2.0
    max_content_length: int = 10000
    column_to_process: str = "RECENT_BLOG"  # <-- Add this line for processing logic

# Initialize config
config = Config()


In [13]:
# Web scraper using requests + BeautifulSoup

import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from typing import List, Dict
from tqdm.notebook import tqdm  # Ensure tqdm is imported

class WebScraper:
    """General-purpose web scraper for different content types."""

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        })
        self.session.verify = True

        self.known_paths = {
            "ABOUT_US": ['/about', '/about-us', '/company', '/who-we-are', '/our-story'],
            "EBOOK": ['/ebook', '/ebooks', '/resources/ebooks'],
            "COURSES": ['/courses', '/training', '/learn', '/classes'],
            "RECENT_BLOG": ['/blog', '/insights', '/news', '/articles'],
            "TESTIMONIALS": ['/testimonials', '/reviews', '/customers'],
            "WEBINAR": ['/webinar', '/webinars', '/events/webinars'],
            "SERVICES": ['/services', '/what-we-do', '/solutions'],
            "PODCAST": ['/podcast', '/podcasts'],
            "SHOP": ['/shop', '/store', '/products']
        }

    def _normalize_url(self, url: str) -> str:
        if not url.startswith(('http://', 'https://')):
            return 'https://' + url
        return url

    def _generate_candidate_urls(self, base_url: str, content_type: str) -> List[str]:
        base_url = self._normalize_url(base_url)
        paths = self.known_paths.get(content_type.upper(), [])
        return [urljoin(base_url, path) for path in paths]

    def _extract_content(self, soup: BeautifulSoup) -> str:
        for element in soup(['script', 'style', 'nav', 'header', 'footer', 'aside', 'form', 'button']):
            element.decompose()

        selectors = [
            'main, .main-content, .content, .page-content',
            'article, .article-content, .post-content',
            '.entry-content, .text-content, .body-content',
            '.container .content, .wrapper .content',
            'section', 'div', 'p'
        ]
        
        for selector in selectors:
            try:
                elements = soup.select(selector)
                if elements:
                    content_parts = [el.get_text(strip=True) for el in elements if el.get_text(strip=True) and len(el.get_text(strip=True)) > 50]
                    content = ' '.join(content_parts)
                    if len(content) > 100:
                        return content[:config.max_content_length]
            except Exception:
                continue

        return "No content found"

    def _try_urls(self, urls: List[str], page_type: str) -> Dict[str, str]:
        for url in urls:
            try:
                response = self.session.get(url, timeout=config.request_timeout, allow_redirects=True)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    content = self._extract_content(soup)
                    if content and content != "No content found" and len(content.strip()) > 100:
                        return {'content': content, 'status': f'success - {page_type} page found', 'found_url': url}
            except Exception:
                pass
            time.sleep(0.3)
        return None

    def scrape_page_by_type(self, base_url: str, content_type: str) -> Dict[str, str]:
        result = {'url': base_url, 'content': '', 'status': 'failed', 'found_url': ''}
        candidate_urls = self._generate_candidate_urls(base_url, content_type)
        result_data = self._try_urls(candidate_urls, content_type)
        
        if result_data:
            result.update(result_data)
        else:
            result['status'] = f'failed - {content_type} not found'
        
        return result

    def scrape_multiple(self, urls: List[str]) -> List[Dict[str, str]]:
        results = []
        for url in tqdm(urls, desc="Scraping websites"):
            result = self.scrape_page_by_type(url, config.column_to_process)
            results.append(result)
            time.sleep(config.delay_between_requests)
            status = "✅" if result['status'].startswith('success') else "❌"
            found_info = f" (found: {result.get('found_url', '').split('/')[-1] or 'homepage'})" if status == "✅" else ""
            print(f"{status} {url}{found_info} - {result['status']}")
        return results


In [14]:
print("🚀 Initializing scraper...")

# Define column mappings
COLUMN_TO_WRITE_URL_TO = {
    "ABOUT_US": "K",
    "EBOOK": "L",
    "COURSES": "M",
    "RECENT_BLOG": "N",
    "TESTIMONIALS": "O",
    "WEBINAR": "P",
    "SERVICES": "Q",
    "PODCAST": "R",
    "SHOP": "S"
}
print("📌 Column mappings configured for categories")

async def process_main_url(row_index: int, main_url: str, category: str, api_key: str):
    """Process a single main URL: crawl, filter suburls, scrape, and prepare updates."""
    print(f"\n🔍 Processing row {row_index}: {main_url}")
    try:
        # Crawl the main URL using Firecrawl
        print(f"🌍 Crawling main URL: {main_url}")
        app = FirecrawlApp(api_key=api_key)
        crawl_result = app.crawl_url(
            main_url,
            limit=5,  # Reduced from 10 to conserve credits
            scrape_options=ScrapeOptions(formats=['html']),  # Simplified to html only
        )
        text = str(crawl_result)
        print(f"✅ Main URL crawled, found content length: {len(text)} characters")
        
        # Extract suburls from the crawl result
        print("🔗 Extracting suburls...")
        parsed_url = urlparse(main_url)
        domain = parsed_url.netloc
        pattern = fr"https?://{re.escape(domain)}[^\s\)\]\(\><'\";=]+"
        all_urls = re.findall(pattern, text)
        junk_chars = '();""=<>'
        clean_urls = [url.rstrip(junk_chars) for url in all_urls]
        site_urls = [
            url for url in clean_urls
            if not url.lower().endswith(('.jpg', '.jpeg', '.png', '.gif', '.svg'))
        ]
        print(f"📈 Found {len(site_urls)} suburls before deduplication")
        unique_urls = list(dict.fromkeys(site_urls))  # Corrected deduplication
        print(f"📈 Deduplicated to {len(unique_urls)} unique suburls")

        # Filter suburls based on category keywords
        categories = {
            "ABOUT_US": ["about", "who-we-are", "company", "our-story", "mission", "vision"],
            "EBOOK": ["ebook", "e-book", "downloads", "whitepaper", "guide", "brochure"],
            "COURSES": ["course", "training", "academy", "learning", "bootcamp"],
            "RECENT_BLOG": ["blog", "insights", "articles", "news", "stories"],
            "TESTIMONIALS": ["testimonial", "reviews", "feedback", "case-studies", "customers"],
            "WEBINAR": ["webinar", "events", "live", "sessions", "recording"],
            "SERVICES": ["service", "solutions", "offerings", "capabilities"],
            "PODCAST": ["podcast", "episodes", "listen", "audio"],
            "SHOP": ["shop", "store", "buy", "product", "checkout", "cart"]
        }
        target_keywords = categories.get(category.upper(), [])
        filtered_urls = [
            url for url in unique_urls
            if any(re.search(kw, url, re.IGNORECASE) for kw in target_keywords)
        ]
        print(f"🔍 Filtered {len(filtered_urls)} suburls for category {category}")

        # Scrape filtered suburls using the WebScraper class
        if filtered_urls:
            scraper = WebScraper()
            results = []
            for url in filtered_urls:
                scraped = scraper.scrape_page_by_type(url, category)
                results.append({
                    "url": scraped["found_url"],
                    "content": scraped["content"],
                    "status": "success" if scraped["status"].startswith("success") else "fail"
                })

            successful_results = [r for r in results if r["status"] == "success"]
            if successful_results:
                content = " ".join(r["content"] for r in successful_results)[:config.max_content_length]
                status = "success" if len(successful_results) == len(results) else "partial success"
                print(f"✅ {main_url} (found: {category.lower()}) - {status} - {len(successful_results)} suburls scraped")
            else:
                content = ""
                status = "fail"
                print(f"❌ {main_url} (found: {category.lower()}) - {status} - no suburls scraped successfully")
        else:
            content = ""
            status = "no suburls"
            print(f"⚠️ {main_url} (found: {category.lower()}) - {status} - no matching suburls found")
    except Exception as e:
        error_msg = str(e).lower()
        if "insufficient credits" in error_msg or "payment required" in error_msg:
            print(f"⚠️ Firecrawl payment error for {main_url}: {e}")
            print("ℹ️ Please check your Firecrawl account at https://firecrawl.dev/pricing")
        else:
            print(f"❌ Error processing {main_url}: {e}")
        content = ""
        status = "error"

    # Prepare updates for Google Sheets
    target_column = COLUMN_TO_WRITE_URL_TO.get(category.upper(), "K")
    content_range = f"{target_column}{row_index}"
    status_range = f"U{row_index}"
    updates = [
        {"range": content_range, "values": [[content]]},
        {"range": status_range, "values": [[status]]}
    ]
    print(f"📝 Prepared updates for row {row_index}: content to {content_range}, status to {status_range}")
    return updates


# Main execution: Process all main URLs and batch update
print(f"\n📊 Processing {len(main_urls_with_rows)} main URLs in 1 batch")
all_updates = []
total_urls = len(main_urls_with_rows)
for idx, (row_index, main_url) in enumerate(main_urls_with_rows, 1):
    # Simulate progress bar
    progress = idx / total_urls
    bar_length = 50
    filled = int(bar_length * progress)
    bar = '█' * filled + '-' * (bar_length - filled)
    percent = progress * 100
    print(f"Scraping websites: {percent:3.0f}% |{bar}| {idx}/{total_urls}")
    
    updates = await process_main_url(row_index, main_url, config.column_to_process, FIRECRAWL_API)
    all_updates.extend(updates)
    print(f"⏳ Delaying for {config.delay_between_requests}s before next URL...")
    time.sleep(config.delay_between_requests)

# Perform batch update to Google Sheets with retries
print("\n📤 Preparing to batch update Google Sheets...")
max_retries = config.max_retries
for attempt in range(max_retries):
    print(f"🔄 Attempt {attempt + 1}/{max_retries} to update Google Sheets")
    try:
        reader.service.spreadsheets().values().batchUpdate(
            spreadsheetId=spreadsheet_id,
            body={"valueInputOption": "RAW", "data": all_updates}
        ).execute()
        print(f"✅ Batch update successful for {len(all_updates)//2} rows")
        break
    except HttpError as e:
        print(f"❌ HTTP Error on attempt {attempt + 1}: {e}")
        if attempt < max_retries - 1:
            wait_time = 5 * (attempt + 1)
            print(f"⏳ Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print("❌ All retries failed.")
    except Exception as e:
        print(f"❌ Error during batch update on attempt {attempt + 1}: {e}")
        if attempt < max_retries - 1:
            print(f"⏳ Retrying in {5 * (attempt + 1)} seconds...")
            time.sleep(5 * (attempt + 1))
        else:
            print("❌ All retries failed.")
print("🏁 Scraper execution complete")

🚀 Initializing scraper...
📌 Column mappings configured for categories

📊 Processing 2 main URLs in 1 batch
Scraping websites:  50% |█████████████████████████-------------------------| 1/2

🔍 Processing row 2: https://www.marcusmillichap.com/
🌍 Crawling main URL: https://www.marcusmillichap.com/
✅ Main URL crawled, found content length: 496088 characters
🔗 Extracting suburls...
📈 Found 307 suburls before deduplication
📈 Deduplicated to 171 unique suburls
🔍 Filtered 32 suburls for category RECENT_BLOG
✅ https://www.marcusmillichap.com/ (found: recent_blog) - success - 32 suburls scraped
📝 Prepared updates for row 2: content to N2, status to U2
⏳ Delaying for 2.0s before next URL...
Scraping websites: 100% |██████████████████████████████████████████████████| 2/2

🔍 Processing row 3: https://www.axiscapital.com/
🌍 Crawling main URL: https://www.axiscapital.com/
✅ Main URL crawled, found content length: 222159 characters
🔗 Extracting suburls...
📈 Found 223 suburls before deduplication
📈 Ded